In [1]:
import pandas as pd

In [2]:
taxi_zone_df = pd.read_csv("taxi_zone_lookup.csv")
taxi_zone_df.head(5)

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
tripdata_df = pd.read_parquet("green_tripdata_2025-11.parquet")
tripdata_df.head(5)

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [5]:
print(tripdata_df.dtypes)

VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag               object
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object


In [17]:
tripdata_df['lpep_pickup_datetime'] = pd.to_datetime(
    tripdata_df['lpep_pickup_datetime']
)

filtered_shorttrip_df = tripdata_df[
    (tripdata_df['lpep_pickup_datetime'] >= '2025-11-01') &
    (tripdata_df['lpep_pickup_datetime'] < '2025-12-01') &
    (tripdata_df['trip_distance'] <= 1.0)
]

len(filtered_shorttrip_df)

8007

In [25]:
filtered_distance_df = tripdata_df[
    tripdata_df['trip_distance'] < 100
]

max_distance = filtered_distance_df['trip_distance'].max()
filtered_distance_df = filtered_distance_df[
    filtered_distance_df['trip_distance'] == max_distance
]

print(filtered_distance_df['lpep_pickup_datetime'].max())

2025-11-14 15:36:27


In [34]:
filtered_data_df = tripdata_df[
    (tripdata_df['lpep_pickup_datetime'] >= '2025-11-18') &
    (tripdata_df['lpep_pickup_datetime'] < '2025-11-19')
]

total_amount_by_pulocation = (
    filtered_data_df
    .groupby('PULocationID')['total_amount']
    .sum()
)

id_max_total_amount = total_amount_by_pulocation.idxmax()

zone_name = taxi_zone_df.loc[taxi_zone_df['LocationID'] == id_max_total_amount, 'Zone'].values[0]

print(zone_name)

East Harlem North


In [48]:
zone_id_pu = taxi_zone_df.loc[taxi_zone_df['Zone'] == 'East Harlem North', 'LocationID'].values[0]

filtered_data_df = tripdata_df[
    (tripdata_df['lpep_pickup_datetime'] >= '2025-11-01') &
    (tripdata_df['lpep_pickup_datetime'] < '2025-12-01') &
    (tripdata_df['PULocationID'] == zone_id_pu)
]

total_tip_by_doLocation = (
    filtered_data_df
    .groupby('DOLocationID', as_index=False)['tip_amount']
    .sum()
    .sort_values(by='tip_amount', ascending=False)
)

print(total_tip_by_doLocation.head(5))

zone_name = taxi_zone_df.loc[taxi_zone_df['LocationID'] == 75, 'Zone'].values[0]

print(zone_name)

     DOLocationID  tip_amount
113           236     4242.01
30             75     3425.94
115           238     2752.60
134           263     2403.17
77            166     2121.65
East Harlem South
